In [ ]:
# === Colab bootstrap (no-op outside Colab) ===========================
# Clones the repo, installs minimal deps, mounts Google Drive, and
# symlinks heavy assets from Drive into the paths the notebook uses.
# See COLAB_SETUP.md for details.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, subprocess, sys
    REPO = "/content/INF8225_Projet"
    if not os.path.isdir(REPO):
        subprocess.check_call([
            "git", "clone", "--depth", "1", "--branch", "temp",
            "https://github.com/moradBMH/INF8225_Projet.git", REPO,
        ])
    if os.getcwd() != REPO:
        os.chdir(REPO)
    if REPO not in sys.path:
        sys.path.insert(0, REPO)
    from colab.setup import setup
    setup()
# ======================================================================

In [ ]:
import os

# Remonte l'arborescence jusqu'à trouver le dossier racine (qui contient 'data' ou 'MedSAM')
while not os.path.isdir('data') and os.getcwd() != os.path.abspath(os.sep):
    os.chdir('..')
print("Dossier de travail actuel :", os.getcwd())

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import torch
from MedSAM.segment_anything import sam_model_registry
from MedSAM.MedSAM_Inference import medsam_inference
from utils import calculate_dice, load_rgb_image, show_segmentation
from colab.drive_paths import output_dir
from skimage import io, transform
from tqdm import tqdm

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Appareil détecté : {device}")

medsam_model = sam_model_registry["vit_b"](checkpoint="MedSAM/work_dir/MedSAM/medsam_vit_b.pth")
medsam_model = medsam_model.to(device)
medsam_model.eval()

In [ ]:
# --- 1. GESTION DES CHEMINS ---
img_folder = "data/Kvasir-SEG/images"
masks_folder = "data/Kvasir-SEG/masks"

results_folder = output_dir("kvasir_implementation", "oracle_baseline_kvasir")
metrics_folder = output_dir("kvasir_implementation", "oracle_baseline_kvasir", "metrics")
output_masks_folder = output_dir(
    "kvasir_implementation", "oracle_baseline_kvasir", "masks", "oracle_segmentation_masks_kvasir"
)
csv_output_path = metrics_folder / "dice_oracle_kvasir.csv"

with open("data/Kvasir-SEG/kavsir_bboxes.json") as f:
    bboxes = json.load(f)

# --- 2. VÉRIFICATION DE L'EXISTENCE DES RÉSULTATS ---
if os.path.exists(csv_output_path):
    print(f"Les résultats existent déjà dans {csv_output_path}.")
    print("Inférence ignorée. Chargement des données existantes...")
    df = pd.read_csv(csv_output_path)
    
else:
    # --- 3. PRÉPARATION DES DONNÉES SI INFÉRENCE NÉCESSAIRE ---
    image_to_split = {}
    for split_name in ['train', 'val', 'test']:
        split_file = f"data/Kvasir-SEG/{split_name}.json"
        if os.path.exists(split_file):
            with open(split_file, 'r') as f:
                coco_data = json.load(f)
                for img_info in coco_data['images']:
                    img_id = os.path.splitext(img_info['file_name'])[0]
                    image_to_split[img_id] = split_name

    dice_list = []
    print("Lancement de la segmentation MedSAM (Oracle)...")

    # --- 4. BOUCLE D'INFÉRENCE ---
    for img in tqdm(os.listdir(img_folder), desc="Segmentation MedSAM"):
        img_path = os.path.join(img_folder, img)
        img_id = os.path.splitext(img)[0]
        
        split = image_to_split.get(img_id, "unknown")

        img_3c = load_rgb_image(img_path)
        H, W, _ = img_3c.shape

        img_1024 = transform.resize(
            img_3c, (1024, 1024), order=3, preserve_range=True, anti_aliasing=True
        ).astype(np.uint8)
        
        img_1024 = (img_1024 - img_1024.min()) / np.clip(
            img_1024.max() - img_1024.min(), a_min=1e-8, a_max=None
        )
        
        img_1024_tensor = (
            torch.tensor(img_1024).float().permute(2, 0, 1).unsqueeze(0).to(device)
        )

        with torch.no_grad():
            image_embedding = medsam_model.image_encoder(img_1024_tensor)

        full_medsam_seg = np.zeros((H, W), dtype=np.uint8)

        if img_id in bboxes:
            for bbox in bboxes[img_id]["bbox"]:
                x_min, y_min = bbox["xmin"], bbox["ymin"]
                x_max, y_max = bbox["xmax"], bbox["ymax"]
            
                box_np = np.array([[x_min, y_min, x_max, y_max]]) 
                box_1024 = box_np / np.array([W, H, W, H]) * 1024

                medsam_seg = medsam_inference(medsam_model, image_embedding, box_1024, H, W)

                full_medsam_seg[medsam_seg > 0] = 1

        io.imsave(
            os.path.join(output_masks_folder, "seg_" + os.path.basename(img_path)),
            (full_medsam_seg*255).astype(np.uint8),
            check_contrast=False,
        )

        mask_path = os.path.join(masks_folder, img)
        if os.path.exists(mask_path):
            true_seg = io.imread(mask_path)
            if len(true_seg.shape) == 3:
                true_seg = true_seg[:,:,0]
            true_seg[true_seg > 0] = 1

            dice_score = calculate_dice(true_seg, full_medsam_seg)

            dice_list.append({
                "image_id": img_id,
                "split": split,
                "dice": dice_score
            })

    df = pd.DataFrame(dice_list)
    df.to_csv(csv_output_path, index=False)
    print(f"\nRésultats sauvegardés dans : {csv_output_path}")

# --- 5. ANALYSE (Exécutée dans tous les cas) ---
print("\n=== STATISTIQUES GLOBALES ORACLE ===")
print(df['dice'].describe())

print("\n=== STATISTIQUES PAR SET ===")
print(df.groupby('split')['dice'].describe())

In [ ]:
show_segmentation(
    img_name="cju2rnkt22xep0801as160g9t.jpg",
    img_folder=img_folder,
    masks_folder=masks_folder,
    output_masks_folder=output_masks_folder,
    bboxes=bboxes
)